In [1]:
import numpy as np
import matplotlib.pyplot as plt
import emcee
import matplotlib
matplotlib.use("Agg")

In [ ]:
def get_fluid_properties(is_saturated):
    if is_saturated:
        k_f = 2.2e9
        g_f = 0.0
        rho_f = 1000 # case where all pores are filled with water, moduli and density of water
    else:
        k_f = 0.0
        g_f = 0.0
        rho_f = 0.02 # case where all pores are filled with air, moduli and density of air
    return k_f, g_f, rho_f

# --- 3. Polarization Factors (P*i and Q*i) ---
def pq_from_alpha(alpha, k_m, mu_m, k_i, mu_i, eps=0.0):
    if alpha <= 0 or alpha > 1:
        return None, None, None

    # Auxiliaries (matrix-based)
    beta_m = mu_m * (3.0*k_m + mu_m) / (3.0*k_m + 4.0*mu_m)
    zeta_m = mu_m * (9.0*k_m + 8.0*mu_m) / (6.0*(k_m + 2.0*mu_m)) # compute the auxiliary variables in table 5

    if alpha <= 1e-1:
        shape = "penny"
        denom_P = k_i + (4.0/3.0)*mu_i + np.pi*alpha*beta_m + eps # eps added to avoid division by zero (if needed)
        P = (k_m + (4.0/3.0)*mu_i) / denom_P
        term1 = 1.0
        term2 = 8.0*mu_m / (4.0*mu_i + np.pi*alpha*(mu_m + 2.0*beta_m) + eps)
        term3 = 2.0 * (k_i + (2.0/3.0)*(mu_i + mu_m)) / (k_i + (4.0/3.0)*mu_i + np.pi*alpha*beta_m + eps) # eps added to avoid division by zero (if needed)
        Q = 0.2 * (term1 + term2 + term3)
    else:
        shape = "sphere"
        P = (k_m + (4.0/3.0)*mu_m) / (k_i + (4.0/3.0)*mu_m + eps)
        Q = (mu_m + zeta_m) / (mu_i + zeta_m + eps)

    return P, Q, shape

# --- 4. Main DEM Numerical Integration Loop ---
def dem_numerical(phi_target, alpha, k_m, mu_m, k_f_sat, g_f_sat, rho_m, rho_f_sat, p_water, step_size=0.001):
    
    # Linear mix fluid properties for partial saturation
    k_i = p_water * k_f_sat
    mu_i = p_water * g_f_sat
    rho_i = p_water * rho_f_sat
    
    # Initialize values at zero porosity (y=0)
    phi = 0.0
    k_star = k_m
    mu_star = mu_m
    if phi_target > 0:
        num_steps = int(phi_target / step_size)
        
        # Simple Euler integration loop
        for _ in range(num_steps):
            # Calculate derivatives at the current step (Eqs. 1 and 2)
            p_star_i, q_star_i, _ = pq_from_alpha(alpha, k_m, mu_m, k_i, mu_i, eps=0.0)
            dk_dy = (k_i - k_star) * p_star_i / (1 - phi)
            dmu_dy = (mu_i - mu_star) * q_star_i / (1 - phi)
            
            # Update K* and G* for the next step
            k_star += dk_dy * step_size
            mu_star += dmu_dy * step_size
            phi += step_size # integration to solve equations 16 and 17
    
    # Final effective moduli and density
    k_eff = k_star
    mu_eff = mu_star
    
    # Calculate final bulk density (assuming mass is conserved)
    rho_eff = (1 - phi_target) * rho_m + phi_target * rho_i
    
    # Placeholder k2 and mubr for consistency with original script's output
    k2 = k_eff
    mubr = mu_eff
    vp = np.sqrt((k_eff + (4/3) * mu_eff) / rho_eff)
    vs = np.sqrt(mu_eff / rho_eff)
    
    return k_eff, mu_eff, vp, vs, rho_eff
    

In [ ]:
#calculate vp, vs,rhob and other parameters from the randomly generated model model parameters
def forward(theta):
    asp = 10**theta[0]
    x_phi = theta[1]
    P_water = theta[2]
    
    # Inputs for the rock physics model
    k_m = theta[3] * 1e9   # Matrix bulk modulus from parameter theta
    mu_m = theta[4] * 1e9   # Matrix shear modulus from parameter theta
    rho_m = theta[-1]      # Matrix density from parameter theta
    
    # Fluid properties for DEM model (these can be defined as global constants
    # or passed as arguments to forward, but this is a simple way to set them up)
    k_f_sat = 2.2e9         # Saturated fluid bulk modulus (e.g., water)
    g_f_sat = 0.0           # Saturated fluid shear modulus (e.g., water)
    rho_f_sat = 1000        # Saturated fluid density (e.g., water)

    # Call the new numerical integration function
    # The return values are renamed to k_eff, g_eff for clarity
    k_eff, g_eff, k2, mubr, rho_eff= dem_numerical(
        phi_target=x_phi, 
        alpha=asp, 
        k_m=k_m, 
        mu_m=mu_m, 
        k_f_sat=k_f_sat, 
        g_f_sat=g_f_sat, 
        rho_m=rho_m, 
        rho_f_sat=rho_f_sat, 
        p_water=P_water
    )

    # Calculate final Vp and Vs from the effective moduli returned by dem_numerical
    vp = np.sqrt((k_eff + (4/3) * g_eff) / rho_eff)
    vs = np.sqrt(g_eff / rho_eff)
    
    out = np.array([vp/(1e3), vs/(1e3), rho_eff])
    return out

In [ ]:
def log_post(theta, lb, ub, d, s, H, prior_sig=0, prior_mu=0, gs=False):
         # shape (3,) for [vp, vs, rho]
    badbb = np.array([np.nan, np.nan, np.nan], dtype=float)
    # 2) Strict bound check (lb < θ ≤ ub)
    if np.any(theta <= lb) or np.any(theta > ub):
        return -np.inf, badbb        # <-- return dM, not None
        # 1) Always compute dM first
    dM = forward(theta)  
    # 3) Physics constraints (vp, vs, rho)
    nH = np.sum(H)
    vp, vs, rho = dM
    if   (nH == 2 and H[0] and H[1] and not (vp>vs)) \
      or (nH == 3            and not (vp>vs)) \
      or (nH == 2 and (H[0] or H[1]) and not (2500 < rho < 3100)) \
      or (nH == 1 and H[2]             and not (2500 < rho < 3100)):
        return -np.inf, dM        # <-- still returns dM

    # 4) Misfit
    misfit = -0.5*np.sum(((d - dM)/s)**2)

    # 5) Optional Gaussian prior
    prior = -0.5*np.sum(((theta-prior_mu)/prior_sig)**2) if gs else 0

    # 6) Final return
    logp = misfit + prior
    return logp, dM

In [ ]:
lb = np.array([-3, 0, 0, 75.6, 25.6, 2680])
ub = np.array([0, 0.5, 1, 80, 40, 2900]) # encoding Table 4 parameter bounds
n = np.shape(ub)[0]
H = np.array([1,1,1], dtype=int)
Ne = 3*n
prior_pdf = np.random.uniform(lb, ub, (Ne, n))
d = np.array([4.1, 2.5, 2589])
s = np.array([0.2, 0.3, 157])
sampler = emcee.EnsembleSampler(Ne,n,log_post,args=(lb, ub, d, s, H))
Nsteps = 50000  #number of MCMC steps
sampler.run_mcmc(prior_pdf, Nsteps, progress=True)
blobs = sampler.get_blobs()

# Extract samples and analyze results
samples = sampler.get_chain(flat=True)

# Plot results
labels = ["Aspect Ratio", "Porosity", "Water Content", "Bulk Modulus", "Shear Modulus", "Density"]
fig, axes = plt.subplots(n, figsize=(10, 7), sharex=True)
for i in range(n):
    axes[i].plot(samples[: , i], "k", alpha=0.3)
    axes[i].set_ylabel(labels[i])
axes[-1].set_xlabel("Step Number")
plt.tight_layout()
plt.show()

100%|████████████████████████████████████| 50000/50000 [04:56<00:00, 168.44it/s]
/var/folders/5p/cy9rmpp11w7ddk5wt_dw0lk00000gp/T/ipykernel_11072/2841635424.py:25: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [7]:
def plot_1d_histograms(samples, labels=None, bins=100, savefig=None):
    n_parameters = samples.shape[1]  
    if labels is None:
        labels = [f"Parameter {i+1}" for i in range(n_parameters)]
    
    fig, axes = plt.subplots(n_parameters, 1, figsize=(8, 2 * n_parameters), sharex=False)
    if n_parameters == 1:
        axes = [axes]
    
    for i in range(n_parameters):
        ax = axes[i]
        ax.hist(samples[:, i], bins=bins, density=True, color='skyblue', edgecolor='black', alpha=0.7)
        ax.set_ylabel("Density")
        ax.set_title(labels[i])
    axes[-1].set_xlabel("Parameter Value")
    
    plt.tight_layout()
    
    plt.show()
    
    if savefig:
        plt.savefig(savefig)
    

In [8]:
label = np.array(['aspect ratio', 'porosity', 'water saturation', 'mineral bulk modulus', 'mineral shear modulus', 'mineral density'])
plot_1d_histograms(samples, labels = label, bins=100, savefig = 'demin_fig_41_1.png')

/var/folders/5p/cy9rmpp11w7ddk5wt_dw0lk00000gp/T/ipykernel_11072/964710158.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [9]:
param_names = list(label)

# Find the indices for porosity and water saturation
idx_porosity = param_names.index('porosity')
idx_water    = param_names.index('water saturation')

# Function to plot and save a single histogram
def plot_and_save(param_idx, param_name, minn, maxx, mayy, bins=100, filename=None):
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.hist(samples[:, param_idx], bins=bins, density=True, alpha=0.7, edgecolor='black')
    #ax.set_xlabel(param_name.capitalize())
    #ax.set_ylabel("Density")
    #ax.set_title(f"Histogram of {param_name.capitalize()}")
    ax.set_xlim(minn, maxx)
    ax.tick_params(
    axis='both',         # apply to both x & y axes
    which='major',       # only affect major ticks
    labelsize=20,        # font size of tick labels
    length=8,            # length of tick marks in points
    width=1.5            # width of the tick marks
    )
    ax.set_ylim(0, mayy)
    plt.tight_layout()
    if filename:
        fig.savefig(filename)
    plt.show()


np.save("saturation_demni.npy", idx_water)
np.save("porosity_demni.npy", idx_porosity)
# Porosity
plot_and_save(idx_porosity, 'crack porosity', 0, 0.5, 13, bins=100, filename='porosity_demin_41.png')

# Water saturation
plot_and_save(idx_water, 'water saturation', 0, 1.0, 6, bins=100, filename='saturation_demin_41.png')

/var/folders/5p/cy9rmpp11w7ddk5wt_dw0lk00000gp/T/ipykernel_11072/979100338.py:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [11]:
bb1 = blobs.reshape(900000, 3)
post_lab = ['vp', 'vs', 'rhob']
plot_1d_histograms(bb1, labels=post_lab, bins=100, savefig = "demin_fig_41_2.png")

/var/folders/5p/cy9rmpp11w7ddk5wt_dw0lk00000gp/T/ipykernel_11072/964710158.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [12]:

def plot_2d_color_plots(samples, lab, savefig=None):
    """
    Plot 2D color maps (histograms) for selected pairs of MCMC parameters.
    
    Parameters:
      samples: numpy array of shape (N, ndim) with MCMC chain samples.
      lab:     list of parameter labels (length ndim).
      savefig: (Optional) Filename to save the figure (e.g., "my_2dplots.png").
    """
    # Extract individual variables from the samples.
    aspect_ratio = samples[:, 0]
    porosity = samples[:, 1]
    water_saturation = samples[:, 2]
    mineral_bulk_modulus = samples[:, 3]
    mineral_shear_modulus = samples[:, 4]
    mineral_density = samples[:, 5]

    # List of variables to plot.
    variables = [aspect_ratio, porosity, water_saturation, mineral_bulk_modulus,
                 mineral_shear_modulus, mineral_density]

    # Define a list of pair indices to plot.
    pairs = [
        (0, 1), (1, 2), (2, 3), (3, 4), (4, 5),  # First 5 pairs.
        (0, 2), (1, 3), (2, 4), (3, 5),          # Next 4 pairs.
        (0, 3), (1, 4), (2, 5),                   # Next 3 pairs.
        (0, 4), (1, 5),                          # Next 2 pairs.
        (0, 5), (1, 0)                           # Last 2 pairs to complete the grid.
    ]
    total_pairs = len(pairs)

    # Create a grid for the subplots.
    nrows = 5
    ncols = 3
    fig, axs = plt.subplots(nrows, ncols, figsize=(15, 15))
    
    plot_counter = 0
    for i in range(nrows):
        for j in range(ncols):
            if plot_counter < total_pairs:
                x_idx, y_idx = pairs[plot_counter]

                # Select the pair of variables to plot.
                x_var = variables[x_idx]
                y_var = variables[y_idx]

                # Compute the 2D histogram with 100 bins per axis.
                hist, x_edges, y_edges = np.histogram2d(x_var, y_var, bins=100)

                # Plot the 2D color map using imshow.
                im = axs[i, j].imshow(hist.T, extent=[x_edges[0], x_edges[-1], y_edges[0], y_edges[-1]],
                                        origin="lower", aspect="auto", cmap="viridis")
                axs[i, j].set_title(f"{lab[y_idx]} vs {lab[x_idx]}")
                axs[i, j].set_xlabel(lab[x_idx])
                axs[i, j].set_ylabel(lab[y_idx])
                
                plot_counter += 1
            else:
                axs[i, j].axis('off')
    
    fig.tight_layout()
    fig.colorbar(im, ax=axs, orientation='horizontal', fraction=0.02, pad=0.04)
    
    
    plt.show()
    if savefig:
        plt.savefig(savefig)
    

# Example usage:
labels = [
    "Aspect Ratio", 
    "Porosity", 
    "Water Saturation", 
    "Matrix Bulk Modulus (Pa)", 
    "Matrix Shear Modulus (Pa)", 
    "Matrix Density (kg/m³)"
]

# Assuming 'samples' is your flattened MCMC chain (numpy array) with 6 columns.
plot_2d_color_plots(samples, labels, savefig="demin_fig_41_3.png")

/var/folders/5p/cy9rmpp11w7ddk5wt_dw0lk00000gp/T/ipykernel_11072/3526005562.py:65: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [13]:
flat_log_prob = sampler.get_log_prob(flat=True)
best_idx = np.argmax(flat_log_prob)
best_params = samples[best_idx]

def W_thickness(S_w, phi):
    return 8500*S_w*phi

print("rough estimate of water layer thickness (m):", W_thickness(best_params[2], best_params[1]))

rough estimate of water layer thickness (m): 2771.866905149415


In [14]:
def marginal_mode(x, bins=100):
    counts, edges = np.histogram(x, bins=bins)
    # find bin with max count, then take its center
    idx = np.argmax(counts)
    return 0.5*(edges[idx] + edges[idx+1])

phi_mode = marginal_mode(samples[:,1], bins=100)
S_w_mode = marginal_mode(samples[:,2], bins=100)
print("Histogram-mode phi:", phi_mode)
print("Histogram-mode S_w:", S_w_mode)
print("Mode-based thickness:", W_thickness(S_w_mode, phi_mode))

Histogram-mode phi: 0.34238759393212365
Histogram-mode S_w: 0.9950067454531248
Mode-based thickness: 2895.7627069363916
